In [ ]:
import pandas as pd
import os

import sys
sys.path.append('..')
from helpers import (
    get_baseline_load_profiles,
    get_rooftop_pv_cf_profiles_by_sector,
    rescale_profile
)

In [3]:
# Create name mappings for EIA-930 respondents, subregions, etc.
hourly_rto_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_rto.csv"
)

hourly_subregion_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_subregion.csv",
    dtype={'subba': str}
)

def get_subbas(ba):
    subbas = hourly_subregion_demand.loc[hourly_subregion_demand.parent == ba].subba.unique().tolist()
    return subbas

In [ ]:
baseline_load_profiles = get_baseline_load_profiles()
rooftop_pv_cf_profiles_by_sector = get_rooftop_pv_cf_profiles_by_sector()

In [6]:
retail_sales = pd.read_csv(
    '../data/county_ftm_sales_estimates.csv',
    index_col=['FIPS']
)
retail_sales.columns = [int(col) for col in retail_sales.columns]

direct_use = pd.read_csv('../data/county_direct_use.csv', index_col=['FIPS', 'is_pv'])
direct_use.columns = [int(col) for col in direct_use.columns]
direct_use = direct_use.groupby(direct_use.index.get_level_values('FIPS')).sum()

distpv_residential_consumption = pd.read_csv(
    '../data/county_distpv_residential_consumption.csv',
    index_col=['FIPS']
)
distpv_residential_consumption.columns = [int(col) for col in distpv_residential_consumption.columns]

distpv_non_residential_consumption = pd.read_csv(
    '../data/county_distpv_non_residential_consumption.csv',
    index_col=['FIPS']
)
distpv_non_residential_consumption.columns = [int(col) for col in distpv_non_residential_consumption.columns]

county_load_percentages = pd.read_csv(
    '../data/county_zone_load_percentage.csv',
    index_col=['FIPS', 'EIAcode']
)
county_load_percentages.columns = [int(col) for col in county_load_percentages.columns]

In [7]:
rsdu = retail_sales.add(direct_use, fill_value=0)

for year in range(2016, 2024):
    rsdu = rsdu.merge(
        county_load_percentages[[year]].rename(columns={year: 'weight'}),
        left_index=True,
        right_index=True
    )
    rsdu[year] *= rsdu['weight']
    rsdu = rsdu.drop(columns='weight')

In [8]:
zone_load_profiles = pd.concat({k:v['value'] for k,v in baseline_load_profiles.items()}, axis=1)

In [ ]:
rsdu_profiles_by_county = {}
for county in rsdu.index.get_level_values('FIPS').unique():
    county_rsdu_by_zone = rsdu.loc[county]
    zones = list(county_rsdu_by_zone.index)
    county_zone_load_profiles = zone_load_profiles[zones].copy()
    county_rsdu_profile = rescale_profile(
        county_zone_load_profiles, county_rsdu_by_zone
    )
    rsdu_profiles_by_county[county] = county_rsdu_profile.sum(axis=1)

county_rsdu_profiles = pd.concat(rsdu_profiles_by_county, axis=1)
county_distpv_residential_consumption_profiles = rescale_profile(
    rooftop_pv_cf_profiles_by_sector['residential'], distpv_residential_consumption
)
county_distpv_non_residential_consumption_profiles = rescale_profile(
    rooftop_pv_cf_profiles_by_sector['commercial'], distpv_non_residential_consumption
)

In [10]:
county_load_profiles = (
    county_rsdu_profiles
    .add(county_distpv_residential_consumption_profiles)
    .add(county_distpv_non_residential_consumption_profiles)
)

In [11]:
os.makedirs('../data/outputs', exist_ok=True)
county_load_profiles.to_hdf(
    '../data/outputs/historic_load_hourly_2016_2023_county.h5',
    key='data'
)